In [5]:
import numpy as np
from scipy.optimize import fsolve


# --- 1. Parameters ---
# Household
beta  = 0.99   # Discount factor
sigma = 2.0    # Risk aversion (intertemporal elasticity of substitution)
phi   = 1.0    # Frisch labor supply elasticity
chi   = 5.6    # Disutility of labor (calibrated for SS labor = 1/3)

# Firm
alpha = 0.33   # Capital share of income
delta = 0.025  # Capital depreciation rate

# Productivity States (Exogenous Shocks)
z_states = np.array([1.02, 0.98])  # [zH, zL]
transition_matrix = np.array([
    [0.9, 0.1],  # From zH to [zH, zL]
    [0.1, 0.9]   # From zL to [zH, zL]
])

# Government
g_y_ratio = 0.2  # G/Y ratio in steady state
initial_B = 0.0  # Initial government debt

# --- 2. Core Model Equations ---

def production_function(K, L, z):
    """Cobb-Douglas Production Function (Section 2.3)"""
    return z * (K**alpha) * (L**(1-alpha))

def marginal_product_labor(K, L, z):
    """Real Wage (w) = MPN"""
    return (1 - alpha) * z * (K**alpha) * (L**(-alpha))

def marginal_product_capital(K, L, z):
    """Rental rate of capital (r + delta) = MPK"""
    return alpha * z * (K**(alpha-1)) * (L**(1-alpha))

def household_utility(C, L):
    """CRRA Utility Function (Section 2.2)"""
    # u(c, l) = (c^(1-sigma))/(1-sigma) - chi * (L^(1+1/phi))/(1+1/phi)
    cons_utility = (C**(1 - sigma)) / (1 - sigma)
    labor_cost   = chi * (L**(1 + 1/phi)) / (1 + 1/phi)
    return cons_utility - labor_cost

# --- 3. Equilibrium Conditions (Section 2.5) ---

def resource_constraint(C, I, G, Y):
    """Market Clearing: Y = C + I + G"""
    return Y - (C + I + G)

def government_budget(G, T, B_prev, B_next, r):
    """G + (1+r)B_prev = T + B_next (Section 2.4)"""
    return G + (1 + r) * B_prev - (T + B_next)

In [6]:
def steady_state_equations(variables):
    K, L, C = variables
    z = 1.0  # Steady state assumes productivity is at its mean
    
    # 1. Output and Prices
    Y = production_function(K, L, z)
    w = marginal_product_labor(K, L, z)
    rk = marginal_product_capital(K, L, z) # rk = r + delta
    
    # 2. Government and Resource Constraint
    G = g_y_ratio * Y
    I = delta * K
    
    # 3. The Equations that must equal ZERO in steady state:
    
    # Condition A: MPK = 1/beta - 1 + delta
    # This comes from the Euler Equation in steady state
    r_target = (1/beta) - 1
    error1 = rk - (r_target + delta)
    
    # Condition B: Intratemporal Choice (Labor Supply vs Wage)
    # The marginal utility of leisure / marginal utility of consumption = real wage
    # For CRRA: chi * L^(1/phi) / C^(-sigma) = w
    error2 = (chi * (L**(1/phi))) / (C**(-sigma)) - w
    
    # Condition C: Resource Constraint (Market Clearing)
    # Y = C + I + G
    error3 = Y - C - I - G
    
    return [error1, error2, error3]

# --- Solve ---
# Initial guesses: K=10, L=0.33, C=1.0
initial_guess = [10.0, 0.33, 1.0]
ss_solution = fsolve(steady_state_equations, initial_guess)

K_ss, L_ss, C_ss = ss_solution
Y_ss = production_function(K_ss, L_ss, 1.0)

print(f"Steady State Results:")
print(f"Labor (L): {L_ss:.4f}")
print(f"Capital (K): {K_ss:.4f}")
print(f"Consumption (C): {C_ss:.4f}")
print(f"Output (Y): {Y_ss:.4f}")

Steady State Results:
Labor (L): 0.4991
Capital (K): 14.1481
Consumption (C): 0.8502
Output (Y): 1.5049
